In [1]:
import matplotlib.pyplot as plt
import numpy as np

In [2]:
EPSILON = 1e-8
def savefig(
        figlist,
        log=True, 
        show_scales=True, 
        y_range:list = None, 
        figsize = (6.4, 4.8),
        label:list = None
    ):
    if (y_range == None):
        y_range = [None for _ in figlist]
    if (label == None):
        label = [None for _ in figlist]
    n = len(figlist)
    # peek into instances
    plt.figure(figsize=figsize)
    
    
    for i, f in enumerate(figlist):
        plt.subplot(n, 1, i+1)
        if (type(f) == tuple):
            for g in f:
                plt.plot(g, scalex=show_scales, scaley=show_scales)
            yrange = y_range[i] or [0, len(g)]
            plt.xlim(yrange)
        elif len(f.shape) == 1:
            plt.plot(f, scalex=show_scales, scaley=show_scales)
            yrange = y_range[i] or [0, len(f)]
            
            plt.xlim(yrange)
        elif len(f.shape) == 2:
            if log:
                x = np.log(f + EPSILON)
            else:
                x = f + EPSILON

            dim_range = y_range[i] or (0, x.shape[0], 0, x.shape[1])
            plt.imshow(x.T, origin='lower', interpolation='none', aspect='auto', extent=dim_range, cmap='magma')
            plt.title(label[i])
            
            # if i==1: plt.ylabel("frequency bin (Hz)")
            # if i==2: plt.xlabel("time frame")

        else:
            raise ValueError('Input dimension must < 3.')
        
        if (not show_scales):
            plt.yticks([])
    
    return plt
    # plt.savefig(filename)
    # plt.close()


def drawmanylines(figlist, figsize = (6.4, 4.8), ylim: bool|list = False):
    plt.figure(figsize=figsize)
    for i, f in enumerate(figlist):
        plt.plot(f)
    plt.xlim([0, len(f)])
    if (type(ylim) != bool):
        plt.ylim(ylim)

def drawmanydots(figlist):
    plt.figure()
    for f in figlist:
        plt.plot(f, 'b.')
    plt.xlim([0, len(f)])
    plt.ylim([
        min(np.min(f), 0),
        np.max(f)
    ])

In [3]:
import pyworld as pw
def test_f0(y, f0, t, sr):
    se = pw.cheaptrick(y, f0, t, sr)
    ap = pw.d4c(y, f0, t, sr)

    ysyn = pw.synthesize(f0, se, ap, sr)

    return ysyn

/home/amogus/miniconda3/envs/preprocess/lib/python3.12/site-packages/pyworld/__init__.py:13: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [4]:
# Get the list of speech
import os

root_folder = '/home/amogus/Research/Major/vqvae/Vu/Data/'
file_list = os.listdir(root_folder)
file_list = [x for x in file_list if x.find('.wav') != -1]

In [6]:
import librosa
from world.harvest_total import harvest

from pystoi import stoi
from pesq import pesq

from tqdm import tqdm

ori_sr = 48000
tgt_sr = 16000

pesq_list = []
stoi_list = []
for audio in tqdm(file_list):
    audio_path =root_folder + audio

    # Read audio
    y, sr = librosa.load(audio_path, mono=False, sr=ori_sr)
    y = librosa.resample(y, orig_sr=ori_sr, target_sr=tgt_sr)[:,:16000]
    ya = np.double(y[0])
    yb = np.double(y[1])

    # Get F0s
    f0a, t = pw.harvest(ya, tgt_sr)
    f0a = pw.stonemask(ya, f0a, t, tgt_sr)
    f0b, t = pw.harvest(yb, tgt_sr)
    f0b = pw.stonemask(yb, f0b, t, tgt_sr)
    f0tot = harvest(yb, tgt_sr )['f0']
    f0tot = pw.stonemask(yb, f0tot, t, tgt_sr)

    # Resynthesize audio
    yasyn = test_f0(ya, f0a, t, tgt_sr)
    ybsyn = test_f0(ya, f0b, t, tgt_sr)
    ytotsyn = test_f0(ya, f0tot, t, tgt_sr)

    try:
        pesq_pair = [
            pesq(tgt_sr, yasyn, ytotsyn),
            pesq(tgt_sr, yasyn, ybsyn)
        ]
        stoi_pair = [
            stoi(yasyn, ytotsyn, tgt_sr),
            stoi(yasyn, ybsyn, tgt_sr)
        ]
    except:
        pesq_pair = [-1, -1]
        stoi_pair = [-1, -1]
    pesq_list.append(pesq_pair)
    stoi_list.append(stoi_pair)


  2%|▏         | 28/1625 [00:36<34:07,  1.28s/it]/home/amogus/miniconda3/envs/preprocess/lib/python3.12/site-packages/pystoi/stoi.py:66: RuntimeWarning: Not enough STFT frames to compute intermediate intelligibility measure after removing silent frames. Returning 1e-5. Please check you wav files
  warnings.warn('Not enough STFT frames to compute intermediate '
100%|██████████| 1625/1625 [35:37<00:00,  1.32s/it]


In [ ]:
# Save the result
# root_eval_folder = './eval/'
# pesq_list = np.array(pesq_list).T
# pesq_ytot = [str(x)+'\n' for x in pesq_list[0]]
# pesq_yb = [str(x)+'\n' for x in pesq_list[1]]
# stoi_list = np.array(stoi_list).T
# stoi_ytot = [str(x)+'\n' for x in stoi_list[0]]
# stoi_yb = [str(x)+'\n' for x in stoi_list[1]]
# with open(root_eval_folder+ 'pesq_ytotsyn.txt', 'w') as f:
#     f.writelines(pesq_ytot)
# with open(root_eval_folder+ 'pesq_ybsyn.txt', 'w') as f:
#     f.writelines(pesq_yb)
# with open(root_eval_folder+ 'stoi_ytotsyn.txt', 'w') as f:
#     f.writelines(stoi_ytot)
# with open(root_eval_folder+ 'stoi_ybsyn.txt', 'w') as f:
#     f.writelines(stoi_yb)

In [16]:
clean_pesq = [x for x in pesq_list.T if x[0] != -1 and x[1] != -1]
clean_stoi = [x for x in stoi_list.T if x[0] != -1 and x[0] != 1e-5 and x[1] != -1 and x[1] != 1e-5]

print(
    np.mean(clean_pesq, axis = 0),
    np.mean(clean_stoi, axis = 0)
)

[2.83473435 3.08786844] [0.98408112 0.9869153 ]
